# 04 - Feature Engineering

**What feature engineering is:** turning raw columns into the specific numeric inputs a model can learn from. Notebooks 02 and 03 already told us *what matters* (day of week, recent weekday averages, events/promos, weather). This notebook turns each of those findings into an actual column: **lags** (past values of the target), **rolling averages** (smoothed recent history), **binary flags** (0/1 indicators), and cleaned-up **weather/calendar** fields.

**One rule that governs every feature here:** a feature must only use information that would have actually been available *before* the date you're forecasting. This is the same leakage rule from notebook 03, just applied column-by-column instead of to the whole train/test split.

## Step 1: Load the full cleaned data

We load the **full** table (including the 2 closed days) first, because the lag features below need to look up "covers on date X" by calendar date — including closed days, which really did have 0 covers. We'll drop closed days from the modeling table at the end, not before.

In [ ]:
import pandas as pd
import numpy as np

full = pd.read_csv("../data/processed/covers_daily_clean.csv", parse_dates=["date"])
full = full.sort_values("date").reset_index(drop=True)

# A calendar-date lookup: covers_by_date[some_date] gives that date's covers (or NaN if unknown)
covers_by_date = full.set_index("date")["covers"]

# Working table: open days only (closed days aren't forecast targets)
df = full[full["is_closed"] == 0].reset_index(drop=True).copy()
print(f"{len(df)} open days to build features for")

## Step 2: Lag features

**Concept:** a lag feature is simply "the target's value N days ago." It's the single most useful kind of feature for covers, because yesterday's/last week's demand is a strong predictor of today's — restaurants don't usually swing wildly overnight.

**Why calendar-date lookup, not `.shift(7)`:** pandas' `.shift()` moves values by *row position*. That's only correct if every date is present with no gaps. Our table has 2 closed days removed, so a plain positional shift would occasionally grab the wrong date. Looking up by actual calendar date (`date - 7 days`) sidesteps that — small dataset, but worth doing right since the same mistake gets worse the more gaps your data has.

**Why 7 and 14, not 1, 2, 3:** which lags are usable depends on your forecast horizon — how far ahead you actually need to predict. If a manager wants next Thursday's forecast on Monday (a common real use case), `covers` from 1–3 days ago won't exist yet at forecast time. Lags of 7 and 14 days (last week / two weeks ago, same weekday) are always available regardless of how far ahead you're forecasting, which is why we stick to those here.

In [ ]:
df["covers_lag_7"] = df["date"].apply(lambda d: covers_by_date.get(d - pd.Timedelta(days=7), np.nan))
df["covers_lag_14"] = df["date"].apply(lambda d: covers_by_date.get(d - pd.Timedelta(days=14), np.nan))

df[["date", "day_of_week", "covers", "covers_lag_7", "covers_lag_14"]].tail(8)

**In Sheets/Excel:** with one row per date sorted ascending, `covers_lag_7` is just `=INDEX(covers_range, MATCH(A2-7, date_range, 0))` — look up the covers value for "this date minus 7," rather than just referencing the cell 7 rows up (which breaks the moment a row is missing, same issue as `.shift()`).

## Step 3: Rolling average features

**Concept:** a rolling average smooths out single-day noise by averaging a recent window. We build two, for two different purposes:

- **`covers_rollavg_4wk_weekday`** — the exact baseline rule from notebook 03 (average of the last 4 occurrences of that weekday), now reused as a *model input* rather than a standalone forecast. Even after adding smarter features, this is often still one of the most useful single columns, since the model can learn how much to trust it versus the other signals.
- **`covers_rollavg_7d_trend`** — a 7-day average across *all* weekdays, shifted back 7 days so it only reflects the week before last (never the days closest to the forecast date). This captures whether the restaurant's overall level is currently running hot or cold — a slow patch or a busy stretch that a single weekday's history wouldn't show on its own.

In [ ]:
def rolling_weekday_avg(target_date, dow, n=4):
    """Average covers from the n most recent same-weekday dates strictly before target_date."""
    past = df[(df["date"] < target_date) & (df["day_of_week"] == dow)].sort_values("date").tail(n)
    return past["covers"].mean() if len(past) else np.nan

df["covers_rollavg_4wk_weekday"] = [
    rolling_weekday_avg(row.date, row.day_of_week) for row in df.itertuples()
]

# 7-day average across all days, computed on the full calendar (incl. closed days), then shifted 7 days back
full_indexed = full.set_index("date")
roll7 = full_indexed["covers"].rolling(7, min_periods=3).mean().shift(7)
df["covers_rollavg_7d_trend"] = df["date"].map(roll7)

df[["date", "day_of_week", "covers", "covers_rollavg_4wk_weekday", "covers_rollavg_7d_trend"]].tail(8)

**In Sheets/Excel:** `covers_rollavg_4wk_weekday` is `=AVERAGEIFS(covers_range, day_of_week_range, A2, date_range, "<"&A2)` combined with only keeping the most recent 4 matches (a helper column ranking recent matches works well here). The 7-day trend is `=AVERAGE(OFFSET(...))` over a 7-row window, then referenced from 7 rows down so it's shifted the same way.

## Step 4: Binary flags and weekend indicator

**Concept:** models work with numbers, not categories or free text, so anything that's naturally "yes/no" should be a clean 0/1 column. `is_weekend`, `is_holiday`, `is_local_event`, and `is_promo` already came out of notebook 01 in exactly this form — the work there of converting the calendar file's text columns into flags is what makes this step trivial now. `day_of_week` itself stays as a category (Monday, Tuesday, ...); the model we fit in notebook 05 will handle turning that into numbers on its own.

One more simplification here: `condition` (Clear/Rain/Storm/Hot-Clear/Cold-Clear) has 5 categories from a Step 4 investigation, which risks the model overfitting to a category with only a handful of days. Since notebook 02 showed precipitation itself has almost no effect on covers, we collapse weather down to two features instead: `temp_high_f` (as-is) and a simple `is_rain` flag.

In [ ]:
df["is_rain"] = (df["precip_in"] > 0).astype(int)

df[["date", "condition", "precip_in", "is_rain", "temp_high_f",
    "is_weekend", "is_holiday", "is_local_event", "is_promo"]].sample(6, random_state=1)

## Step 5: Assemble the model-ready table

Pick the final feature columns plus the target (`covers`), then drop rows with any missing feature. Two sources of missing values are expected and fine to drop:

1. **The first ~2 weeks of history** — there's no data 7 or 14 days earlier yet, so lag/rolling features are undefined.
2. **A handful of dates downstream of the 4 originally-missing target days from notebook 01** — we deliberately left those as NaN rather than inventing a value, so any lag/rolling feature that depends on one of them is honestly NaN too, not a bug.

In [ ]:
feature_cols = [
    "date", "day_of_week", "is_weekend", "month", "week_of_year",
    "covers_lag_7", "covers_lag_14",
    "covers_rollavg_4wk_weekday", "covers_rollavg_7d_trend",
    "is_holiday", "is_local_event", "is_promo",
    "temp_high_f", "is_rain",
    "covers",
]

model_df = df[feature_cols].copy()
print(f"Before dropping incomplete rows: {len(model_df)}")

model_df = model_df.dropna().reset_index(drop=True)
print(f"After dropping incomplete rows: {len(model_df)}")

model_df.head()

## Step 6: Save the model-ready dataset

In [ ]:
model_df.to_csv("../data/processed/covers_model_features.csv", index=False)
print("Saved to data/processed/covers_model_features.csv")
model_df.dtypes

## Recap

Built four categories of features, each tied to a finding from notebooks 02–03:

- **Lags** (`covers_lag_7`, `covers_lag_14`) — recent same-weekday demand, looked up by calendar date so gaps in the data can't silently shift them.
- **Rolling averages** (`covers_rollavg_4wk_weekday`, `covers_rollavg_7d_trend`) — the notebook 03 baseline itself, plus an overall momentum signal, both computed walk-forward so nothing leaks from the future.
- **Binary flags** (`is_weekend`, `is_holiday`, `is_local_event`, `is_promo`, `is_rain`) — the exact levers notebook 02 showed move covers, and the ones a manager can see coming on a calendar.
- **Weather** (`temp_high_f`) — kept as-is; its effect is real but modest, per notebook 02's correlation check.

694 of the 728 open days survived with every feature present — the rest were dropped only because there wasn't enough history yet or they trace back to originally-missing target days.

**Next up:** the time-based train/test split (same idea as notebook 03, now applied to this richer feature table), fitting a real model, generating test-period forecasts, and checking whether it actually beats the 19.0 MAE / 12.9% MAPE baseline.